In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
import re
import copy
import torchvision

In [2]:
synth_ds = datasets.ImageFolder(
    "/workspace/alvin/SAR_ML/data/SAMPLE/png_images/qpm/synth",
    transforms.ToTensor()
)
meas_ds = datasets.ImageFolder(
    "/workspace/alvin/SAR_ML/data/SAMPLE/png_images/qpm/real",
    transforms.ToTensor()
)

In [3]:
def extract_elev(path):
    match = re.search(r"elevDeg_(\d{3})", path)
    if match:
        return int(match.group(1))
    return None

def filter_by_elev(dataset, allowed_angles):
    filtered = []
    ds_cp = copy.deepcopy(dataset)
    for path, label in dataset.samples:
        elev = extract_elev(path)
        if elev in allowed_angles:
            filtered.append((path, label))
    ds_cp.samples = filtered
    ds_cp.targets = [label for _, label in filtered]
    return ds_cp

In [4]:
train_ds = filter_by_elev(synth_ds, {14, 15, 16})
test_ds  = filter_by_elev(meas_ds, {17})

In [5]:
ds_dict = {"train" : train_ds, "test": test_ds}
dataset_sizes = {"train" : len(train_ds), "test": len(test_ds)}

dataloaders = {x : DataLoader(ds_dict[x], batch_size = 32, num_workers = 4, shuffle = True) for x in ds_dict.keys()}

In [6]:
for i in range(10):
    print(f"Training Run {i}")
    # load pre-trained model
    model = models.resnet18(weights = "DEFAULT")
    # Replace final layer for the number of classes
    model.fc = nn.Linear(model.fc.in_features, len(train_ds.class_to_idx))
    
    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss() # most common used nn for classification problems
    
    optimizer = optim.Adam(model.parameters(), lr = 10**(-3))
    
    scheduler = CosineAnnealingLR(optimizer, T_max=100)
    
    # move model to GPU
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    history = {
        "train_loss": [],
        "train_acc": []
    }
    
    # Training loops
    num_epochs = 100
    for epoch in range(num_epochs):
        print(f"Epoch {epoch}")
        for phase in ["train"]:
            if phase == "train":
                model.train()
            else:
                model.eval()
    
            running_loss = 0.0
            running_corrects = 0 # correct predictions
    
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
    
                optimizer.zero_grad() # clear the gradient from previous iteration
    
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels) # check if output and labels match
    
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                        # scheduler.step() # scheduler here if OneCycleLR
    
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
    
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc.item())
    
            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
    
        scheduler.step()
        print(f"Epoch {epoch} LR: {scheduler.get_last_lr()[0]:.10f}")

        if epoch == 0:
            torch.save(copy.deepcopy(model.state_dict()), f"/workspace/alvin/SAR_ML/weights/rn18_run{i}_epoch0.pth")
        
    print("Training complete!")

    torch.save(model.state_dict(), f"/workspace/alvin/SAR_ML/weights/rn18_run{i}.pth")

Training Run 0
Epoch 0
train Loss: 0.6912 Acc: 0.7543
Epoch 0 LR: 0.0009997533
Epoch 1
train Loss: 0.1128 Acc: 0.9628
Epoch 1 LR: 0.0009990134
Epoch 2
train Loss: 0.0399 Acc: 0.9901
Epoch 2 LR: 0.0009977810
Epoch 3
train Loss: 0.0225 Acc: 0.9938
Epoch 3 LR: 0.0009960574
Epoch 4
train Loss: 0.0597 Acc: 0.9826
Epoch 4 LR: 0.0009938442
Epoch 5
train Loss: 0.0502 Acc: 0.9888
Epoch 5 LR: 0.0009911436
Epoch 6
train Loss: 0.0455 Acc: 0.9864
Epoch 6 LR: 0.0009879584
Epoch 7
train Loss: 0.0400 Acc: 0.9913
Epoch 7 LR: 0.0009842916
Epoch 8
train Loss: 0.0576 Acc: 0.9814
Epoch 8 LR: 0.0009801468
Epoch 9
train Loss: 0.1091 Acc: 0.9628
Epoch 9 LR: 0.0009755283
Epoch 10
train Loss: 0.0137 Acc: 0.9950
Epoch 10 LR: 0.0009704404
Epoch 11
train Loss: 0.0023 Acc: 1.0000
Epoch 11 LR: 0.0009648882
Epoch 12
train Loss: 0.0064 Acc: 0.9975
Epoch 12 LR: 0.0009588773
Epoch 13
train Loss: 0.0324 Acc: 0.9888
Epoch 13 LR: 0.0009524135
Epoch 14
train Loss: 0.0123 Acc: 0.9950
Epoch 14 LR: 0.0009455033
Epoch 15
train 

In [45]:
for i in range(10):
    new_model = models.resnet18(weights = None) # dont load ImageNet Weights
    new_model.fc = nn.Linear(new_model.fc.in_features, len(labels_map))
    
    # Load your trained weights
    new_model.load_state_dict(torch.load(
        "/mnt/d/Users/Admin/Projects/Machine_Learning/weights/fashionmnist_epoch_50_resnet18_onecycle.pth",
        map_location=device
    ))
    
    new_model = new_model.to(device)
    new_model.eval()
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
    
            outputs = new_model(inputs)
            _, preds = torch.max(outputs, 1)
    
            correct += torch.sum(preds == labels).item()
            total += labels.size(0)
    
    test_acc = correct / total
    print(f"Test Accuracy: {test_acc:.4f}")

'/workspace/alvin/SAR_ML/notebooks'